# Análisis estadístico avanzado — Windows

Notebook para revisar tomografía, tejido adiposo, corteza motora, normalización, comparación entre pacientes y aproximación al sano.

Lee resultados desde:

```bat
D:\EAFIT\01-2026\proyecto\resultados
```

Guarda salidas en la carpeta de análisis estadístico avanzado de Windows.


In [ ]:
%matplotlib inline
from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
try:
    from scipy import stats
    SCIPY_OK=True
except Exception:
    SCIPY_OK=False
warnings.filterwarnings('ignore')
RESULTS_ROOT = Path(r'D:\EAFIT\01-2026\proyecto\resultados')
OUT_DIR = RESULTS_ROOT / 'analisis_estadistico_avanzado_windows'
GRAF_DIR = OUT_DIR / 'graficos'
OUT_DIR.mkdir(parents=True, exist_ok=True)
GRAF_DIR.mkdir(parents=True, exist_ok=True)
SANO='sano'
PACIENTES=['paciente 3','paciente 6','paciente 7','paciente 9']
print('RESULTS_ROOT:', RESULTS_ROOT)
print('OUT_DIR:', OUT_DIR)
print('SciPy:', SCIPY_OK)

## 1. Funciones de normalización y clasificación

In [ ]:
def norm_txt(x):
    s=str(x).strip()
    for a,b in {'á':'a','é':'e','í':'i','ó':'o','ú':'u','Á':'A','É':'E','Í':'I','Ó':'O','Ú':'U'}.items():
        s=s.replace(a,b)
    return s

def norm_stage(x):
    s=norm_txt(x).lower()
    if s in ['antes','pre','before','baseline']: return 'Antes'
    if s in ['despues','post','after','seguimiento']: return 'Despues'
    return str(x)

def norm_subject(x):
    s=norm_txt(x).lower().strip()
    if s in ['sano','control','healthy']: return 'sano'
    m=re.search(r'(?:paciente\s*)?(\d+)',s)
    if m: return f'paciente {int(m.group(1))}'
    return str(x).strip()

def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(',','.',regex=False).str.replace('nan','',regex=False).str.strip(), errors='coerce')

def infer_subject(path):
    for p in Path(path).parts:
        low=norm_txt(p).lower()
        if low=='sano': return 'sano'
        if low.startswith('paciente '): return norm_subject(p)
    return 'desconocido'

def infer_stage(path):
    for p in Path(path).parts:
        st=norm_stage(p)
        if st in ['Antes','Despues']: return st
    return 'desconocido'

def infer_modality(path):
    s='/'.join(norm_txt(p).lower() for p in Path(path).parts)
    for key,val in [('tomografia','tomografia'),('morfometria','morfometria'),('resonancias','resonancias'),('emg','emg'),('dinamometria','dinamometria'),('mapas','mapas'),('estructura_funcion','estructura_funcion'),('correlaciones','correlaciones'),('consolidado','consolidado')]:
        if key in s: return val
    return 'general'

def metric_family(name, modality=''):
    n=norm_txt(name).lower()
    if ('corteza' in n or 'motor' in n or 'motora' in n) and any(k in n for k in ['volumen','volume','voxels','ml','cm3']): return 'volumen_corteza_motora'
    if any(k in n for k in ['fat_to_muscle','grasa_musculo','grasa/musculo','total_fat_to_muscle','intramuscular_fat_to_muscle','subcutaneous_fat_to_muscle']): return 'relacion_grasa_musculo'
    if any(k in n for k in ['muscle_volume','volumen_muscular','global_muscle','segmented_muscle_volume']): return 'volumen_muscular'
    if any(k in n for k in ['fat_volume','grasa','subcutaneous','intramuscular']): return 'grasa'
    if any(k in n for k in ['dice','jaccard','overlap','pearson','spearman','corr']): return 'similitud'
    if any(k in n for k in ['rmse','mae','error','distancia']): return 'error_distancia'
    return 'otras'

def orientation(name, family):
    n=norm_txt(name).lower()
    if family in ['relacion_grasa_musculo','grasa','error_distancia'] or any(k in n for k in ['rmse','mae','dvars','fd']): return 'lower'
    if family=='similitud' or any(k in n for k in ['dice','jaccard','pearson','spearman','tsnr']): return 'higher'
    return 'context'

## 2. Cargar resultados consolidados, tomografía avanzada y reportes TXT

In [ ]:
def load_long_csvs(root):
    files=list(root.rglob('*_todos_los_datos_largo.csv')) + list(root.rglob('resumen_metricas_tomografia_avanzada.csv')) + list(root.rglob('resumen_global_tomografia_avanzada.csv')) + list(root.rglob('resumen_global_correccion_corteza.csv'))
    rows=[]
    for f in sorted(set(files)):
        try: d=pd.read_csv(f, low_memory=False)
        except Exception: continue
        if 'metric_name' not in d.columns:
            num_cols=[c for c in d.columns if to_num(d[c]).notna().sum()>0]
            if not num_cols: continue
            idcols=[c for c in d.columns if c not in num_cols]
            d=d.melt(id_vars=idcols, value_vars=num_cols, var_name='metric_name', value_name='metric_value')
        if 'metric_value' not in d.columns: continue
        if 'subject' not in d.columns: d['subject']=infer_subject(f)
        if 'patient' not in d.columns: d['patient']=d['subject']
        if 'stage' not in d.columns: d['stage']=infer_stage(f)
        if 'modality' not in d.columns: d['modality']=infer_modality(f)
        if 'source_file' not in d.columns: d['source_file']=str(f)
        d['metric_value']=to_num(d['metric_value'])
        d=d.dropna(subset=['metric_value'])
        rows.append(d)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def parse_tomo_txt(report_path):
    text=Path(report_path).read_text(encoding='utf-8', errors='ignore')
    rows=[]; subject=infer_subject(report_path); stage=infer_stage(report_path); section=''; entity='global'
    keymap={'Relación grasa/músculo':'fat_to_muscle_ratio','Relación músculo/grasa':'muscle_to_fat_ratio','Grasa respecto a grasa + músculo':'fat_fraction_percent','Área de grasa subcutánea':'subcutaneous_fat_area_cm2','Área muscular total':'total_muscle_area_cm2','Volumen muscular segmentado':'segmented_muscle_volume_cm3','Volumen de grasa intramuscular':'intramuscular_fat_volume_cm3','Relación grasa IM / volumen músculo':'intramuscular_fat_to_muscle_ratio','Volumen muscular total del lado':'side_muscle_volume_cm3','Volumen grasa intramuscular':'side_intramuscular_fat_volume_cm3','Volumen grasa subcutánea':'side_subcutaneous_fat_volume_cm3','Volumen total de grasa':'side_total_fat_volume_cm3','Relación grasa IM / músculo':'side_intramuscular_fat_to_muscle_ratio','Relación grasa subcutánea / músculo':'side_subcutaneous_fat_to_muscle_ratio','Relación grasa total / músculo':'side_total_fat_to_muscle_ratio','Volumen muscular total':'global_muscle_volume_cm3','Volumen total de grasa intramuscular':'global_intramuscular_fat_volume_cm3','Volumen grasa subcutánea / músculos':'global_subcutaneous_fat_volume_cm3','Volumen total de grasa':'global_total_fat_volume_cm3','Distancia entre landmarks':'landmark_distance_mm'}
    for line in text.splitlines():
        l=line.strip(); up=l.upper()
        if not l: continue
        if 'COMPOSICIÓN CORPORAL' in up: section='composicion_corte_medio'
        elif 'GRASA INTRAMUSCULAR' in up: section='grasa_intramuscular'
        elif 'RESUMEN POR MIEMBRO' in up: section='resumen_por_miembro'
        elif 'RESUMEN GLOBAL' in up: section='resumen_global'; entity='global'
        if l.startswith('MIEMBRO '): entity=l.replace('MIEMBRO ','').strip().lower().capitalize(); continue
        if l.startswith('MÚSCULO ') or l.startswith('MUSCULO '): entity=l.split(' ',1)[1].strip(); continue
        if l.startswith('-') and ':' in l:
            k,v=l.lstrip('-').split(':',1); k=k.strip(); v=v.strip()
            m=re.search(r'[-+]?\d+(?:[\.,]\d+)?', v)
            if not m: continue
            rows.append({'subject':subject,'patient':subject,'stage':stage,'modality':'tomografia_avanzada_txt','section':section,'entity':entity,'metric_name':keymap.get(k, re.sub(r'[^A-Za-z0-9_]+','_',norm_txt(k).lower()).strip('_')),'metric_value':float(m.group(0).replace(',','.')),'source_file':str(report_path)})
    return pd.DataFrame(rows)

main_df=load_long_csvs(RESULTS_ROOT)
text_rows=[parse_tomo_txt(p) for p in RESULTS_ROOT.rglob('Reporte_Morfometrico_Completo.txt')]
text_df=pd.concat([x for x in text_rows if not x.empty], ignore_index=True) if any(not x.empty for x in text_rows) else pd.DataFrame()
raw=pd.concat([main_df, text_df], ignore_index=True) if not main_df.empty or not text_df.empty else pd.DataFrame()
if raw.empty: raise RuntimeError('No encontré datos numéricos en resultados')
raw['subject']=raw['subject'].map(norm_subject); raw['stage']=raw['stage'].map(norm_stage); raw['metric_name']=raw['metric_name'].astype(str); raw['metric_value']=pd.to_numeric(raw['metric_value'], errors='coerce')
raw=raw.dropna(subset=['metric_value']); raw['family']=[metric_family(a,b) for a,b in zip(raw['metric_name'], raw['modality'])]; raw['orientation']=[orientation(a,b) for a,b in zip(raw['metric_name'], raw['family'])]
raw.to_csv(OUT_DIR/'00_datos_largos_integrados.csv', index=False, encoding='utf-8-sig')
print('Datos cargados:', raw.shape)
display(raw.head(20))

## 3. Normalización para evitar sesgos de escala

In [ ]:
def add_normalization(df):
    d=df.copy(); grp=d.groupby('metric_name')['metric_value']
    d['z_global']=grp.transform(lambda x: (x-x.mean())/(x.std(ddof=0) if x.std(ddof=0)>1e-12 else np.nan))
    d['robust_z_global']=grp.transform(lambda x: (x-x.median())/((x.quantile(.75)-x.quantile(.25))/1.349 if (x.quantile(.75)-x.quantile(.25))>1e-12 else np.nan))
    sano=d[d.subject==SANO]; sano_ref=sano[sano.stage=='Despues'] if not sano.empty and not sano[sano.stage=='Despues'].empty else sano[sano.stage=='Antes']
    if not sano_ref.empty:
        sm=sano_ref.groupby('metric_name')['metric_value'].mean().rename('sano_mean'); ss=sano_ref.groupby('metric_name')['metric_value'].std().replace(0,np.nan).rename('sano_sd')
        d=d.merge(sm,on='metric_name',how='left').merge(ss,on='metric_name',how='left')
        d['ratio_vs_sano']=d.metric_value/d.sano_mean.replace(0,np.nan); d['z_vs_sano']=(d.metric_value-d.sano_mean)/d.sano_sd; d['abs_distance_vs_sano']=abs(d.metric_value-d.sano_mean)
    else:
        d['sano_mean']=np.nan; d['sano_sd']=np.nan; d['ratio_vs_sano']=np.nan; d['z_vs_sano']=np.nan; d['abs_distance_vs_sano']=np.nan
    return d

data=add_normalization(raw)
data.to_csv(OUT_DIR/'01_datos_normalizados.csv', index=False, encoding='utf-8-sig')
display(data.head(20))

## 4. Descriptiva avanzada

In [ ]:
desc=(data.groupby(['subject','stage','modality','family','metric_name'],dropna=False)['metric_value'].agg(n='count',mean='mean',std='std',median='median',min='min',q25=lambda x:np.nanpercentile(x,25),q75=lambda x:np.nanpercentile(x,75),max='max').reset_index())
desc['cv_pct']=100*desc['std']/desc['mean'].replace(0,np.nan)
desc.to_csv(OUT_DIR/'02_descriptiva_avanzada.csv', index=False, encoding='utf-8-sig')
display(desc.head(30))

## 5. Cambios Antes vs Después y recuperación hacia el sano

In [ ]:
rows=[]
for (sub,mod,met),g in data[data.subject!=SANO].groupby(['subject','modality','metric_name'],dropna=False):
    a=g[g.stage=='Antes'].metric_value.dropna().to_numpy(float); b=g[g.stage=='Despues'].metric_value.dropna().to_numpy(float)
    if len(a)==0 or len(b)==0: continue
    ma,mb=float(np.mean(a)),float(np.mean(b)); delta=mb-ma; dp=100*delta/ma if abs(ma)>1e-12 else np.nan; fam=metric_family(met,mod); ori=orientation(met,fam)
    p=np.nan; u=np.nan
    if SCIPY_OK and len(a)>=2 and len(b)>=2:
        try: p=float(stats.ttest_ind(b,a,equal_var=False).pvalue)
        except Exception: pass
        try: u=float(stats.mannwhitneyu(b,a,alternative='two-sided').pvalue)
        except Exception: pass
    pooled=np.nan
    if len(a)>1 and len(b)>1:
        pooled=np.sqrt((np.var(a,ddof=1)+np.var(b,ddof=1))/2)
    hedges=(mb-ma)/pooled if np.isfinite(pooled) and pooled>1e-12 else np.nan
    mejora=np.nan if ori=='context' else ((delta>0) if ori=='higher' else (delta<0))
    rows.append({'subject':sub,'modality':mod,'metric_name':met,'family':fam,'orientation':ori,'n_antes':len(a),'n_despues':len(b),'mean_antes':ma,'mean_despues':mb,'delta':delta,'delta_pct':dp,'hedges_g':hedges,'p_welch':p,'p_mannwhitney':u,'mejora_directa_seg_orientacion':mejora})
change=pd.DataFrame(rows)
if not data[data.subject==SANO].empty:
    sane=data[(data.subject==SANO)&(data.stage=='Despues')]
    if sane.empty: sane=data[(data.subject==SANO)&(data.stage=='Antes')]
    sm=sane.groupby(['modality','metric_name'])['metric_value'].mean().rename('sano_mean').reset_index(); pm=data[data.subject!=SANO].groupby(['subject','stage','modality','metric_name'])['metric_value'].mean().reset_index(); pre=pm[pm.stage=='Antes'].rename(columns={'metric_value':'mean_antes'}).drop(columns=['stage']); post=pm[pm.stage=='Despues'].rename(columns={'metric_value':'mean_despues'}).drop(columns=['stage']); rec=pre.merge(post,on=['subject','modality','metric_name']).merge(sm,on=['modality','metric_name'])
    rec['dist_antes_sano']=abs(rec.mean_antes-rec.sano_mean); rec['dist_despues_sano']=abs(rec.mean_despues-rec.sano_mean); rec['mejora_distancia_sano']=rec.dist_antes_sano-rec.dist_despues_sano; rec['indice_recuperacion_hacia_sano']=rec.mejora_distancia_sano/rec.dist_antes_sano.replace(0,np.nan); rec['despues_mas_cerca_sano']=rec.indice_recuperacion_hacia_sano>0; rec['family']=[metric_family(a,b) for a,b in zip(rec.metric_name,rec.modality)]
else:
    rec=pd.DataFrame()
change.to_csv(OUT_DIR/'03_cambios_antes_vs_despues.csv',index=False,encoding='utf-8-sig'); rec.to_csv(OUT_DIR/'04_recuperacion_hacia_sano_por_metrica.csv',index=False,encoding='utf-8-sig')
display(change.head(20)); display(rec.head(20))

## 6. Volumen de corteza motora como métrica del protocolo

In [ ]:
cortex=data[data.family=='volumen_corteza_motora'].copy()
if cortex.empty:
    display(Markdown('No encontré métricas clasificadas como volumen de corteza motora.'))
    cortex_summary=pd.DataFrame()
else:
    cortex_summary=(cortex.groupby(['stage','metric_name'],dropna=False)['metric_value'].agg(n='count',mean='mean',std='std',median='median',min='min',max='max').reset_index())
    cortex_summary['cv_pct']=100*cortex_summary['std']/cortex_summary['mean'].replace(0,np.nan)
    cortex_summary['protocol_stability_score']=1-(cortex_summary['cv_pct']/100)
    cortex_summary.to_csv(OUT_DIR/'05_protocolo_volumen_corteza_motora.csv',index=False,encoding='utf-8-sig')
    display(cortex_summary.sort_values('cv_pct').head(30))

## 7. Relación grasa/músculo y composición corporal

In [ ]:
fatmus=data[data.family.isin(['relacion_grasa_musculo','volumen_muscular','grasa'])].copy()
if fatmus.empty:
    display(Markdown('No encontré métricas de grasa/músculo. Corre la tomografía avanzada primero.'))
    fat_summary=pd.DataFrame()
else:
    fat_summary=(fatmus.groupby(['subject','stage','family','metric_name'],dropna=False)['metric_value'].agg(n='count',mean='mean',std='std',median='median').reset_index())
    fat_summary.to_csv(OUT_DIR/'06_resumen_grasa_musculo.csv',index=False,encoding='utf-8-sig')
    display(fat_summary.head(40))

## 8. Resumen por paciente y gráficos

In [ ]:
if rec.empty:
    resumen=pd.DataFrame(); resumen_global=pd.DataFrame()
else:
    resumen=(rec.groupby(['subject','family']).agg(n_metricas=('metric_name','count'),pct_acercan_sano=('despues_mas_cerca_sano',lambda x:100*np.mean(x)),media_indice_recuperacion=('indice_recuperacion_hacia_sano','mean'),mediana_indice_recuperacion=('indice_recuperacion_hacia_sano','median')).reset_index())
    resumen_global=(rec.groupby('subject').agg(n_metricas=('metric_name','count'),pct_acercan_sano=('despues_mas_cerca_sano',lambda x:100*np.mean(x)),media_indice_recuperacion=('indice_recuperacion_hacia_sano','mean'),mediana_indice_recuperacion=('indice_recuperacion_hacia_sano','median')).reset_index())
    resumen.to_csv(OUT_DIR/'07_resumen_por_paciente_y_familia.csv',index=False,encoding='utf-8-sig'); resumen_global.to_csv(OUT_DIR/'08_resumen_global_por_paciente.csv',index=False,encoding='utf-8-sig')

def save_show(name):
    path=GRAF_DIR/name; plt.tight_layout(); plt.savefig(path,dpi=180,bbox_inches='tight'); plt.show(); display(Markdown(f'Guardado: `{path}`')); plt.close()

if not resumen_global.empty:
    d=resumen_global.sort_values('media_indice_recuperacion'); plt.figure(figsize=(9,5)); y=np.arange(len(d)); plt.barh(y,d.media_indice_recuperacion); plt.yticks(y,d.subject); plt.axvline(0,ls='--'); plt.axvline(1,ls=':'); plt.xlabel('Índice recuperación hacia sano'); plt.title('Recuperación global hacia sano'); save_show('01_recuperacion_global_hacia_sano.png')
if not rec.empty:
    piv=rec.groupby(['subject','family'])['indice_recuperacion_hacia_sano'].mean().reset_index().pivot(index='subject',columns='family',values='indice_recuperacion_hacia_sano')
    fig,ax=plt.subplots(figsize=(max(10,1.1*len(piv.columns)),4+0.5*len(piv))); im=ax.imshow(piv.to_numpy(float),aspect='auto'); ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns,rotation=45,ha='right'); ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index); fig.colorbar(im,label='Índice recuperación'); ax.set_title('Recuperación hacia sano por familia')
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            val=piv.iloc[i,j]
            if np.isfinite(val): ax.text(j,i,f'{val:.2f}',ha='center',va='center',fontsize=8)
    save_show('02_heatmap_recuperacion_familias.png')
if not fatmus.empty:
    d=fatmus[fatmus.family=='relacion_grasa_musculo'].groupby(['subject','stage'])['metric_value'].mean().reset_index()
    if not d.empty:
        plt.figure(figsize=(9,5))
        for sub,g in d.groupby('subject'):
            g=g.set_index('stage').reindex(['Antes','Despues']).reset_index(); plt.plot(g.stage,g.metric_value,marker='o',label=sub)
        plt.ylabel('Relación grasa/músculo promedio'); plt.title('Cambio grasa/músculo Antes vs Después'); plt.legend(); save_show('03_grasa_musculo_antes_despues.png')
if not cortex.empty:
    d=cortex.groupby(['subject','stage'])['metric_value'].mean().reset_index(); plt.figure(figsize=(9,5))
    for sub,g in d.groupby('subject'):
        g=g.set_index('stage').reindex(['Antes','Despues']).reset_index(); plt.plot(g.stage,g.metric_value,marker='o',label=sub)
    plt.ylabel('Volumen corteza motora / voxeles promedio'); plt.title('Estabilidad de volumen de corteza motora'); plt.legend(); save_show('04_volumen_corteza_motora_estabilidad.png')
if not change.empty:
    top=change.copy(); top['abs_g']=top.hedges_g.abs(); top=top.sort_values('abs_g',ascending=False).head(25); labels=(top.subject+' | '+top.family+' | '+top.metric_name).str.slice(0,80)
    plt.figure(figsize=(12,max(6,.35*len(top)))); y=np.arange(len(top)); plt.barh(y,top.hedges_g); plt.yticks(y,labels); plt.axvline(0,ls='--'); plt.xlabel('Hedges g Después - Antes'); plt.title('Mayores cambios Antes vs Después'); plt.gca().invert_yaxis(); save_show('05_top_cambios_antes_despues.png')

## 9. Conclusiones automáticas y exportación

In [ ]:
concs=[]
if not resumen_global.empty:
    for _,r in resumen_global.iterrows():
        idx=r.media_indice_recuperacion; pct=r.pct_acercan_sano
        cls='recuperación alta hacia sano' if idx>=0.75 else 'recuperación moderada hacia sano' if idx>=0.40 else 'recuperación leve hacia sano' if idx>0.05 else 'sin cambio global claro frente al sano' if idx>=-0.05 else 'se aleja del sano'
        sub=r.subject; cortex_txt=''; fat_txt=''
        if not cortex.empty:
            c=cortex[cortex.subject==sub]
            if not c.empty: cortex_txt=f" Volumen corteza motora medio={c.metric_value.mean():.3f}; CV interno={100*c.metric_value.std()/c.metric_value.mean() if abs(c.metric_value.mean())>1e-12 else np.nan:.2f}%."
        if not fatmus.empty:
            fm=fatmus[(fatmus.subject==sub)&(fatmus.family=='relacion_grasa_musculo')]
            if not fm.empty: fat_txt=f" Relación grasa/músculo media={fm.metric_value.mean():.4f}."
        concs.append({'subject':sub,'clasificacion':cls,'pct_metricas_acercan_sano':pct,'indice_recuperacion':idx,'conclusion':f"{sub}: {cls}. {pct:.1f}% de métricas se acercan al sano; índice medio {idx:.3f}.{cortex_txt}{fat_txt}"})
conclusiones=pd.DataFrame(concs); conclusiones.to_csv(OUT_DIR/'09_conclusiones_avanzadas.csv',index=False,encoding='utf-8-sig')
with pd.ExcelWriter(OUT_DIR/'analisis_estadistico_avanzado_completo.xlsx',engine='openpyxl') as w:
    data.to_excel(w,'datos_normalizados',index=False); desc.to_excel(w,'descriptiva',index=False); change.to_excel(w,'antes_vs_despues',index=False); rec.to_excel(w,'recuperacion_sano',index=False); resumen_global.to_excel(w,'resumen_global',index=False); resumen.to_excel(w,'resumen_familia',index=False); cortex_summary.to_excel(w,'protocolo_corteza',index=False); fat_summary.to_excel(w,'grasa_musculo',index=False); conclusiones.to_excel(w,'conclusiones',index=False)
report='# Reporte estadístico avanzado\n\n' + '\n'.join('- '+c for c in conclusiones.get('conclusion',[]))
(OUT_DIR/'REPORTE_ESTADISTICO_AVANZADO.md').write_text(report,encoding='utf-8')
display(conclusiones); print('Salidas:', OUT_DIR)

## 10. Galería final

In [ ]:
imgs=sorted(GRAF_DIR.glob('*.png'))
display(Markdown(f'### {len(imgs)} gráficos guardados'))
for p in imgs:
    display(Markdown(f'#### {p.name}'))
    display(Image(filename=str(p)))